# Análisis de Retrasos y Operaciones de Vuelos

# Mi proyecto · De la teoría a la práctica (Sesiones 1 a 3)
**Plantilla espejo del caso guiado Olist** · Módulo: Preparación, Recopilación y Análisis Exploratorio de Datos · MIADAS – UMSA

**Grupo:** _escribe aquí los integrantes_
**Conjunto de datos:** _Flight Delay Dataset — 2024_ · **Fuente y URL:** _https://www.kaggle.com/datasets/hrishitpatil/flight-data-2024_

Esta plantilla tiene **las mismas secciones** que el caso guiado. Repite cada paso con **tu conjunto de datos**. Este notebook evolucionará en las próximas sesiones con la limpieza, el análisis exploratorio, la normalización y la calidad.

### Requisitos mínimos de tu conjunto de datos
- [x] Al menos **1 000 registros**
- [x] Al menos **8 variables** que mezclen escalas (nominal, ordinal, de razón)
- [x] Al menos **una variable de fecha**
- [x] **Problemas de calidad** reales (faltantes, duplicados o inconsistencias)
- [x] Idealmente, **dos tablas o fuentes** que se puedan integrar
- [x] **Licencia** que permita su uso académico

In [27]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [28]:
!pip install matplotlib seaborn


[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Paso 0 · El conjunto de datos y cómo lo obtenemos
**Flight Delay Dataset — 2024.** Datos de vuelos comerciales de EE. UU. del año 2024, originados en el *Bureau of Transportation Statistics* (BTS) y publicados en Kaggle. Trabajamos con una **muestra de 10 000 vuelos**, dividida en **tres tablas** que se relacionan por `flight_id`.

- **Fuente:** Kaggle (`kaggle.com/datasets/hrishitpatil/flight-data-2024`), datos originales del BTS.
- **Formato:** CSV, uno por tabla.
- **Acceso:** leemos cada archivo directamente desde el **repositorio de GitHub del grupo miadas-grupo4-proyecto** con `pd.read_csv(url)`. No hace falta descargarlo.

Usaremos tres tablas: **schedule** (planificación del vuelo), **performance** (desempeño operacional) y **delays** (retrasos y causas).

> **Plan B (si falla la conexión):** usa los CSV de la carpeta `data/` del repositorio y cambia cada dirección por, por ejemplo, `"data/flight_schedule.csv"`.

In [29]:
schedule    = pd.read_csv("https://raw.githubusercontent.com/corinaccqf/miadas-grupo4-proyecto/corina/data/flight_schedule.csv")
performance = pd.read_csv("https://raw.githubusercontent.com/corinaccqf/miadas-grupo4-proyecto/corina/data/flight_performance.csv")
delays      = pd.read_csv("https://raw.githubusercontent.com/corinaccqf/miadas-grupo4-proyecto/corina/data/flight_delays.csv")

print(schedule.shape, performance.shape, delays.shape)

(10000, 18) (10000, 14) (10000, 8)


In [30]:
print("schedule", schedule.shape)
print("performance: ", performance.shape)
print("delays ", delays.shape)

schedule (10000, 18)
performance:  (10000, 14)
delays  (10000, 8)


**Resultado esperado:** schedule (10000, 18) · performance (10000, 14) · delays (10000, 8)

se puede obseervar que las tres tablas tienen el mismo numero de filas

---
# S1 · De la Sesión 1: tipos de datos y escalas de medición
**Recordemos:** Stevens (1946) propuso cuatro escalas — **nominal, ordinal, intervalo y razón** — y la escala determina qué cálculos son válidos.

### S1.1 · Primera mirada a la planificación de los vuelos
`head()` muestra las primeras filas e `info()` el tipo de dato de cada columna.

In [31]:
performance.head(3)

,flight_id,dep_time,dep_delay,taxi_out,wheels_off,wheels_on,taxi_in,arr_time,arr_delay,cancelled,cancellation_code,diverted,actual_elapsed_time,air_time
0,1,1015.0,-3.0,21.0,1036.0,1135.0,4.0,1139.0,-10.0,0,NaN,0,144.0,119.0
1,2,1633.0,-4.0,14.0,1647.0,1900.0,6.0,1906.0,-17.0,0,NaN,0,273.0,253.0
2,3,952.0,-8.0,13.0,1005.0,1034.0,8.0,1042.0,-17.0,0,NaN,0,50.0,29.0


In [32]:
schedule.head(3)


,flight_id,year,month,day_of_month,day_of_week,fl_date,op_unique_carrier,op_carrier_fl_num,origin,origin_city_name,origin_state_nm,dest,dest_city_name,dest_state_nm,crs_dep_time,crs_arr_time,crs_elapsed_time,distance
0,1,2024,4,18,4,2024-04-18,MQ,3535,DFW,"Dallas/Fort Worth, TX",Texas,RAP,"Rapid City, SD",South Dakota,1018,1149,151,835
1,2,2024,1,1,1,2024-01-01,AA,148,CLT,"Charlotte, NC",North Carolina,PHX,"Phoenix, AZ",Arizona,1637,1923,286,1773
2,3,2024,12,12,4,2024-12-12,9E,5440,CHA,"Chattanooga, TN",Tennessee,ATL,"Atlanta, GA",Georgia,1000,1059,59,106


In [33]:
delays.head(3)

,flight_id,dep_delay,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,1,-3.0,-10.0,0,0,0,0,0
1,2,-4.0,-17.0,0,0,0,0,0
2,3,-8.0,-17.0,0,0,0,0,0


In [34]:
schedule.info()
delays.info()
performance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   flight_id          10000 non-null  int64 
 1   year               10000 non-null  int64 
 2   month              10000 non-null  int64 
 3   day_of_month       10000 non-null  int64 
 4   day_of_week        10000 non-null  int64 
 5   fl_date            10000 non-null  object
 6   op_unique_carrier  10000 non-null  object
 7   op_carrier_fl_num  10000 non-null  int64 
 8   origin             10000 non-null  object
 9   origin_city_name   10000 non-null  object
 10  origin_state_nm    10000 non-null  object
 11  dest               10000 non-null  object
 12  dest_city_name     10000 non-null  object
 13  dest_state_nm      10000 non-null  object
 14  crs_dep_time       10000 non-null  int64 
 15  crs_arr_time       10000 non-null  int64 
 16  crs_elapsed_time   10000 non-null  int64 

## Cambiamos el tipo de Dato
**Qué observar:** segun se puede observar la fecha del vuelo (`fl_date`) aparece como `str`), no como fecha. Mientras una fecha sea texto, no podemos restar fechas ni agrupar por semana. La **convertimos** con `pd.to_datetime`:

In [35]:
schedule["fl_date"] = pd.to_datetime(schedule["fl_date"])

In [36]:
print(schedule["fl_date"].dtype)

datetime64[ns]


In [37]:
print(schedule["fl_date"])

0      2024-04-18
1      2024-01-01
2      2024-12-12
3      2024-04-08
4      2024-02-16
          ...    
9995   2024-01-16
9996   2024-07-05
9997   2024-02-28
9998   2024-02-18
9999   2024-03-01
Name: fl_date, Length: 10000, dtype: datetime64[ns]


## Eliminar columnas innecesarias

**Qué hacemos:** quitamos las columnas `year`, `month`, `day_of_month` y `day_of_week` de la tabla `schedule`.

**Por qué:**
- `year` vale siempre 2024, así que no aporta información.
- `month`, `day_of_month` y `day_of_week` ya están dentro de `fl_date`, por lo que son datos repetidos.

Si las necesitamos más adelante, las podemos sacar de `fl_date`.

In [38]:
# Drop columns we do not need
schedule = schedule.drop(columns=["year", "month", "day_of_month", "day_of_week"])

In [39]:
import math
import pandas as pd

print(schedule.shape)
schedule.head()

(10000, 14)


,flight_id,fl_date,op_unique_carrier,op_carrier_fl_num,origin,origin_city_name,origin_state_nm,dest,dest_city_name,dest_state_nm,crs_dep_time,crs_arr_time,crs_elapsed_time,distance
0,1,2024-04-18,MQ,3535,DFW,"Dallas/Fort Worth, TX",Texas,RAP,"Rapid City, SD",South Dakota,1018,1149,151,835
1,2,2024-01-01,AA,148,CLT,"Charlotte, NC",North Carolina,PHX,"Phoenix, AZ",Arizona,1637,1923,286,1773
2,3,2024-12-12,9E,5440,CHA,"Chattanooga, TN",Tennessee,ATL,"Atlanta, GA",Georgia,1000,1059,59,106
3,4,2024-04-08,WN,1971,OMA,"Omaha, NE",Nebraska,LAS,"Las Vegas, NV",Nevada,1330,1430,180,1099
4,5,2024-02-16,WN,862,BWI,"Baltimore, MD",Maryland,MYR,"Myrtle Beach, SC",South Carolina,1340,1510,90,399


In [40]:

schedule["flight_id"] = schedule["flight_id"].astype(str)
schedule["op_unique_carrier"] = schedule["op_unique_carrier"].astype(str)
schedule["op_carrier_fl_num"] = schedule["op_carrier_fl_num"].astype(str)
schedule["origin"] = schedule["origin"].astype(str)
schedule["origin_city_name"] = schedule["origin_city_name"].astype(str)
schedule["origin_state_nm"] = schedule["origin_state_nm"].astype(str)
schedule["dest"] = schedule["dest"].astype(str)
schedule["dest_city_name"] = schedule["dest_city_name"].astype(str)
schedule["dest_state_nm"] = schedule["dest_state_nm"].astype(str)

### S1.3 · Escala de cada variable
Completa la tabla con tus variables (mínimo 8):

| Variable | Escala (nominal / ordinal / intervalo / razón) | Cálculos válidos |
|---|---|---|
|flight_id|Nominal|Frecuencias, conteo|
|year|Intervalo|Diferencias entre años|
|month|Ordinal|Frecuencias, porcentajes, moda|
|day_of_month|Intervalo|Media, mediana, moda|
|day_of_week|Nominal|recuencias, moda|
|fl_date|Intervalo|Diferencias entre fechas|
|op_unique_carrier|Nominal|Frecuencias, porcentajes, moda|
|op_carrier_fl_num|Nominal|Frecuencias|
|origin|Nominal|Frecuencias, porcentajes, moda|
|dest|Nominal|Frecuencias, porcentajes, moda|
|crs_dep_time|Intervalo|Diferencias entre horarios|
|crs_dep_time|Intervalo|Diferencias entre horarios|

In [41]:
# Frecuencias de una variable nominal u ordinal:
schedule["flight_id"].value_counts()


flight_id
10000    1
1        1
2        1
3        1
4        1
        ..
20       1
21       1
22       1
23       1
24       1
Name: count, Length: 10000, dtype: int64

In [42]:
# Estadísticos de una variable de razón:
schedule[["crs_elapsed_time", "distance"]].describe().round(2)

,crs_elapsed_time,distance
count,10000.00,10000.00
mean,147.37,838.35
std,72.93,598.53
min,23.00,31.00
25%,93.00,404.00
50%,130.00,680.00
75%,179.00,1072.00
max,685.00,5095.00


---
# S2 · Muestreo (Sesión 2)
¿Qué subconjunto necesitarías revisar a mano o encuestar? Calcula el tamaño de muestra y compara una muestra aleatoria con una estratificada.

In [43]:
def tamano_muestra(N, e=0.05, z=1.96, p=0.5):
    n0 = z**2 * p * (1 - p) / e**2
    return math.ceil(n0 / (1 + (n0 - 1) / N))

n = tamano_muestra(len(schedule))
n

370

In [44]:
variable_estrato = "op_unique_carrier"
aleatoria = schedule.sample(n=n, random_state=42)
estratificada = schedule.groupby(variable_estrato, group_keys=False).sample(frac=n / len(schedule), random_state=42)
pd.DataFrame({
    "Población": schedule[variable_estrato].value_counts(normalize=True),
    "Aleatoria": aleatoria[variable_estrato].value_counts(normalize=True),
    "Estratificada": estratificada[variable_estrato].value_counts(normalize=True),
}).round(3)

,Población,Aleatoria,Estratificada
op_unique_carrier,,,
9E,0.028,0.024,0.027
AA,0.138,0.124,0.138
AS,0.032,0.038,0.033
B6,0.033,0.043,0.033
DL,0.146,0.135,0.146
F9,0.030,0.035,0.030
G4,0.017,0.008,0.016
HA,0.010,0.011,0.011
MQ,0.041,0.054,0.041


---
# S3 · Fuentes e integración (Sesión 3)
Si tu conjunto tiene dos o más tablas, únelas por su clave. Si tiene una sola, identifica qué fuente externa podría complementarla (por ejemplo, datos del INE).

In [45]:
BASE = "https://raw.githubusercontent.com/corinaccqf/miadas-grupo4-proyecto/corina/data/"

performance = pd.read_csv(BASE + "flight_performance.csv")
performance["flight_id"] = performance["flight_id"].astype(str)
datos_performance = schedule.astype({"flight_id": str}).merge(performance, on="flight_id", how="left")

delay = pd.read_csv(BASE + "flight_delays.csv")
delay["flight_id"] = delay["flight_id"].astype(str)
datos_performance_delay = schedule.astype({"flight_id": str}).merge(delay, on="flight_id", how="left")

print(datos_performance_delay.shape)

(10000, 21)


---
# Cierre · Informe de recolección inicial (CRISP-DM, fase 2)

| Elemento | Mi conjunto de datos |
|---|---|
| **Fuente** | |
| **Tipo de fuente** (interna / externa · primaria / secundaria) | |
| **Formato y acceso** (CSV, Excel, API… · URL) | |
| **Tablas y número de registros** | |
| **Periodo que cubre** | |
| **Problemas encontrados** | |
| **Primer hallazgo o pregunta de negocio** | |